### Задание

Попробуйте поработать с датасетом юридических текстов. В датасете всего две важных колонки признаков: заголовок дела и его текст, а целевая переменная - case_outcome (мультиклассовая классификация). 

В базовом варианте можно оставить только текст дела, если хотите поинтереснее - можно попробовать распарсить case_title, добыв оттуда дополнительные признаки. 

https://www.kaggle.com/datasets/amohankumar/legal-text-classification-dataset

In [56]:
import pandas as pd
import numpy as np
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn import metrics

from sklearn.metrics import classification_report

from nltk.tokenize import word_tokenize

from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

In [57]:
data = pd.read_csv('/Users/katyamazurina/Desktop/2 sem/ML/legal_text_classification.csv')
data.head()

,case_id,case_outcome,case_title,case_text
0,Case1,cited,Alpine Hardwood (Aust) Pty Ltd v Hardys Pty Lt...,Ordinarily that discretion will be exercised s...
1,Case2,cited,Black v Lipovac [1998] FCA 699 ; (1998) 217 AL...,The general principles governing the exercise ...
2,Case3,cited,Colgate Palmolive Co v Cussons Pty Ltd (1993) ...,Ordinarily that discretion will be exercised s...
3,Case4,cited,Dais Studio Pty Ltd v Bullett Creative Pty Ltd...,The general principles governing the exercise ...
4,Case5,cited,Dr Martens Australia Pty Ltd v Figgins Holding...,The preceding general principles inform the ex...


In [37]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24985 entries, 0 to 24984
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   case_id       24985 non-null  object
 1   case_outcome  24985 non-null  object
 2   case_title    24985 non-null  object
 3   case_text     24809 non-null  object
dtypes: object(4)
memory usage: 780.9+ KB


In [58]:
data = data.dropna() #убираем пустые значения
data = data.drop('case_id', axis=1) #case_id тоже уберу

In [59]:
#разделяем данные на трейн и тест
x_train, x_test, y_train, y_test = train_test_split(data.case_text, data.case_outcome)

In [60]:
#униграммы
vec = CountVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(x_train)

list(vec.vocabulary_.items())[:10]

[('since', 36141),
 ('the', 39158),
 ('applicant', 6665),
 ('based', 7907),
 ('his', 20495),
 ('submission', 37567),
 ('that', 39155),
 ('privilege', 31591),
 ('had', 19664),
 ('been', 8131)]

In [22]:
print(bow.shape) #размер матрицы признаков

(18606, 43082)


In [61]:
#обучаем логистисческую регрессию
clf = LogisticRegression(solver='liblinear', random_state=42)
clf.fit(bow, y_train)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LogisticRegression(random_state=42, solver='liblinear')

In [62]:
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

               precision    recall  f1-score   support

     affirmed       0.32      0.64      0.43        14
      applied       0.15      0.31      0.20       296
     approved       0.09      0.30      0.14        10
        cited       0.90      0.58      0.70      4667
   considered       0.16      0.33      0.22       205
    discussed       0.17      0.37      0.24       116
distinguished       0.23      0.60      0.33        65
     followed       0.12      0.39      0.18       165
  referred to       0.30      0.50      0.38       655
      related       0.23      0.80      0.36        10

     accuracy                           0.54      6203
    macro avg       0.27      0.48      0.32      6203
 weighted avg       0.73      0.54      0.60      6203



In [63]:
#логистичсекая регрессия на би- и триграммах, ограничу количество признаков
vec = CountVectorizer(ngram_range=(2, 3), max_features=43082)
bow = vec.fit_transform(x_train)

list(vec.vocabulary_.items())[:10]

[('since the', 32029),
 ('the applicant', 34400),
 ('his submission', 17550),
 ('submission that', 32625),
 ('that privilege', 33691),
 ('privilege had', 29122),
 ('had been', 16485),
 ('been waived', 8814),
 ('upon the', 40868),
 ('the disclosure', 35312)]

In [26]:
print(bow.shape) #матрица признаков достаточно большая, поэтому ограничу количество признаков для би- и триграмм до 43082

(18606, 2863657)


In [64]:
clf = LogisticRegression(solver='liblinear', random_state=42)
clf.fit(bow, y_train)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LogisticRegression(random_state=42, solver='liblinear')

In [65]:
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

               precision    recall  f1-score   support

     affirmed       0.18      0.56      0.27         9
      applied       0.22      0.35      0.27       391
     approved       0.12      0.44      0.19         9
        cited       0.81      0.66      0.73      3761
   considered       0.24      0.38      0.29       257
    discussed       0.28      0.40      0.33       174
distinguished       0.28      0.51      0.36        92
     followed       0.31      0.47      0.37       354
  referred to       0.52      0.49      0.51      1145
      related       0.26      0.82      0.39        11

     accuracy                           0.57      6203
    macro avg       0.32      0.51      0.37      6203
 weighted avg       0.64      0.57      0.60      6203



In [66]:
#TF-IDF
vec = TfidfVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(x_train)
clf = LogisticRegression(solver='liblinear', random_state=42)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

               precision    recall  f1-score   support

     affirmed       0.07      0.67      0.13         3
      applied       0.05      0.31      0.09       107
     approved       0.00      0.00      0.00         0
        cited       0.96      0.53      0.69      5448
   considered       0.03      0.37      0.06        35
    discussed       0.05      0.57      0.09        21
distinguished       0.01      1.00      0.02         2
     followed       0.08      0.55      0.13        77
  referred to       0.27      0.57      0.37       510
      related       0.00      0.00      0.00         0

     accuracy                           0.53      6203
    macro avg       0.15      0.46      0.16      6203
 weighted avg       0.87      0.53      0.64      6203



/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation

In [68]:
noise = stopwords.words('english') + list(punctuation)

In [69]:
vec = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize, stop_words=noise)
bow = vec.fit_transform(x_train)
clf = LogisticRegression(solver='liblinear', random_state=42)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'m", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


               precision    recall  f1-score   support

     affirmed       0.46      0.68      0.55        19
      applied       0.20      0.35      0.25       361
     approved       0.12      0.44      0.19         9
        cited       0.83      0.64      0.72      3951
   considered       0.20      0.33      0.25       244
    discussed       0.24      0.39      0.29       152
distinguished       0.25      0.55      0.34        76
     followed       0.29      0.43      0.35       369
  referred to       0.46      0.50      0.48      1005
      related       0.31      0.65      0.42        17

     accuracy                           0.57      6203
    macro avg       0.34      0.50      0.39      6203
 weighted avg       0.65      0.57      0.60      6203

